# 🚗 Tesla OTA Update Classifier - BERT Pipeline v2
**Bachelor Thesis - Amine**

Ce notebook entraîne 3 modèles BERT pour classifier automatiquement les fonctions des mises à jour Tesla :
- **Domain** (Powertrain, Chassis, Body, HMI/Multimedia, Driving Automation)
- **Function Type** (NEW, MODIFIED, REMOVED)
- **Coordination** (STANDALONE, COORD_ACTIVE)

In [ ]:
# STEP 1: Install dependencies
!pip install transformers torch openpyxl pandas scikit-learn -q
print('✅ Dependencies installed!')

In [ ]:
# STEP 2: Upload your Excel files
from google.colab import files
import os
print('Please upload your 2 files:')
print('1. 20260512_Pilot_Study_Draft_Amine__1_.xlsx')
print('2. 20260525_Amine_First_Scrap.xlsx')
uploaded = files.upload()

# Fix filename if needed
for fname in os.listdir('/content/'):
    if 'Pilot_Study' in fname and fname != '20260512_Pilot_Study_Draft_Amine__1_.xlsx':
        os.rename(fname, '20260512_Pilot_Study_Draft_Amine__1_.xlsx')
        print(f'Renamed {fname}')
    if 'First_Scrap' in fname and fname != '20260525_Amine_First_Scrap.xlsx':
        os.rename(fname, '20260525_Amine_First_Scrap.xlsx')
        print(f'Renamed {fname}')

In [ ]:
# STEP 3: Load and prepare data
import pandas as pd
from openpyxl import load_workbook

wb_coding = load_workbook('20260512_Pilot_Study_Draft_Amine__1_.xlsx', read_only=True)
ws_coding = wb_coding['INTERMEDIARY_FUNCTIONAL_CODING']
rows_coding = list(ws_coding.iter_rows(values_only=True))

coding_data = []
for row in rows_coding[1:]:
    function_id = row[0]
    update_id = row[1]
    domain = row[7]
    function_name = row[8]
    function_type = row[9]
    coord = row[13]
    if function_id and domain and function_type and function_name:
        domain_norm = domain.strip()
        if 'HMI' in domain_norm:
            domain_norm = 'HMI/Multimedia/Telematics'
        coding_data.append({
            'function_id': function_id,
            'update_id': update_id,
            'domain_primary': domain_norm,
            'function_name': str(function_name),
            'function_type': function_type,
            'standalone_or_coord': coord
        })

df = pd.DataFrame(coding_data)
print(f'Total coded functions: {len(df)}')
print(df['domain_primary'].value_counts())
print(df['function_type'].value_counts())
print(df['standalone_or_coord'].value_counts())

In [ ]:
# STEP 4: Setup BERT
import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import Dataset, DataLoader
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

print('GPU available:', torch.cuda.is_available())
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

le_domain = LabelEncoder()
df['domain_encoded'] = le_domain.fit_transform(df['domain_primary'])
print(f'Domain classes: {list(le_domain.classes_)}')

df_type = df[df['function_type'] != 'REMOVED'].copy()
le_type = LabelEncoder()
df_type['type_encoded'] = le_type.fit_transform(df_type['function_type'])
print(f'Type classes: {list(le_type.classes_)}')

df_coord = df[df['standalone_or_coord'].isin(['STANDALONE', 'COORD_ACTIVE'])].copy()
le_coord = LabelEncoder()
df_coord['coord_encoded'] = le_coord.fit_transform(df_coord['standalone_or_coord'])
print(f'Coord classes: {list(le_coord.classes_)}')

tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
print('✅ BERT tokenizer loaded!')

In [ ]:
# STEP 5: Dataset class
class TeslaDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=64):
        self.texts = list(texts)
        self.labels = list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_tensors='pt'
        )
        return {
            'input_ids': encoding['input_ids'].squeeze(),
            'attention_mask': encoding['attention_mask'].squeeze(),
            'label': torch.tensor(self.labels[idx], dtype=torch.long)
        }

In [ ]:
# STEP 6: Training function
def train_bert(texts, labels, num_labels, label_encoder, name='Model', epochs=10, lr=2e-5):
    X_train, X_test, y_train, y_test = train_test_split(
        list(texts), list(labels), test_size=0.2, random_state=42, stratify=labels
    )

    train_dataset = TeslaDataset(X_train, y_train, tokenizer)
    test_dataset = TeslaDataset(X_test, y_test, tokenizer)
    train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
    test_loader = DataLoader(test_dataset, batch_size=16)

    model = BertForSequenceClassification.from_pretrained(
        'bert-base-uncased', num_labels=num_labels
    ).to(device)

    optimizer = torch.optim.AdamW(model.parameters(), lr=lr)

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        for batch in train_loader:
            optimizer.zero_grad()
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels_batch = batch['label'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels_batch)
            loss = outputs.loss
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f'Epoch {epoch+1}/{epochs} - Loss: {total_loss/len(train_loader):.4f}')

    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for batch in test_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            outputs = model(input_ids, attention_mask=attention_mask)
            preds = torch.argmax(outputs.logits, dim=1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(batch['label'].numpy())

    acc = accuracy_score(all_labels, all_preds)
    print(f'\n{name} Accuracy: {acc:.2%}')
    print(classification_report(all_labels, all_preds, target_names=label_encoder.classes_))
    return model, acc

In [ ]:
# STEP 7: Train Domain Classifier
print('=== Training BERT Domain Classifier ===')
domain_model, domain_acc = train_bert(
    df['function_name'], df['domain_encoded'],
    len(le_domain.classes_), le_domain, 'Domain Classifier', epochs=10
)

In [ ]:
# STEP 8: Train Function Type Classifier
print('=== Training BERT Function Type Classifier ===')
type_model, type_acc = train_bert(
    df_type['function_name'], df_type['type_encoded'],
    len(le_type.classes_), le_type, 'Type Classifier', epochs=10
)

In [ ]:
# STEP 9: Train Coordination Classifier
print('=== Training BERT Coordination Classifier ===')
coord_model, coord_acc = train_bert(
    df_coord['function_name'], df_coord['coord_encoded'],
    len(le_coord.classes_), le_coord, 'Coordination Classifier', epochs=10
)

In [ ]:
# STEP 10: Test predictions
def predict(model, le, text):
    model.eval()
    encoding = tokenizer(text, max_length=64, padding='max_length', truncation=True, return_tensors='pt')
    input_ids = encoding['input_ids'].to(device)
    attention_mask = encoding['attention_mask'].to(device)
    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        pred = torch.argmax(outputs.logits, dim=1).item()
    return le.inverse_transform([pred])[0]

test_examples = [
    'Full Self-Driving Beta', 'Camp Mode', 'Supercharger Improvements',
    'Navigate on Autopilot', 'Regenerative Braking', 'Cabin Camera',
    'Suspension Improvements', 'Maps Improvements', 'Dynamic Brake Lights',
    'Hey Grok', 'Tire Pressure Warning', 'Smart Summon',
]

print(f"{'Function':<35} {'Domain':<30} {'Type':<12} {'Coord':<15}")
print('-'*92)
for fname in test_examples:
    d = predict(domain_model, le_domain, fname)
    t = predict(type_model, le_type, fname)
    c = predict(coord_model, le_coord, fname)
    print(f"{fname:<35} {d:<30} {t:<12} {c:<15}")

In [ ]:
# STEP 11: Save and download all models
import pickle

torch.save(domain_model.state_dict(), 'domain_bert_model.pt')
torch.save(type_model.state_dict(), 'type_bert_model.pt')
torch.save(coord_model.state_dict(), 'coord_bert_model.pt')

with open('domain_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_domain, f)
with open('type_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_type, f)
with open('coord_label_encoder.pkl', 'wb') as f:
    pickle.dump(le_coord, f)

print(f'✅ All models saved!')
print(f'Domain: {domain_acc:.2%} | Type: {type_acc:.2%} | Coord: {coord_acc:.2%}')

files.download('domain_bert_model.pt')
files.download('type_bert_model.pt')
files.download('coord_bert_model.pt')
files.download('domain_label_encoder.pkl')
files.download('type_label_encoder.pkl')
files.download('coord_label_encoder.pkl')
print('✅ All models downloaded!')